# Advanced Python Slicing Problems with Solutions

This notebook contains advanced practice problems on Python slicing, `slice()` objects, negative steps, slice normalization with `.indices()`, fixed-width parsing, and custom sequence behavior.

Each problem is followed by a complete solution and runnable checks.

## Problem 1 — Predict the Slice Result

Given the sequence below, predict the result of each slice **without running the code first**.

```python
data = list(range(10))

a = data[2:8:2]
b = data[8:2:-2]
c = data[-2:-8:-2]
d = data[3:-1:-1]
e = data[3::-1]
f = data[::-3]
```

Explain why `d` is empty.

In [1]:
# Solution 1

data = list(range(10))

a = data[2:8:2]
b = data[8:2:-2]
c = data[-2:-8:-2]
d = data[3:-1:-1]
e = data[3::-1]
f = data[::-3]

results = {
    "a": a,
    "b": b,
    "c": c,
    "d": d,
    "e": e,
    "f": f,
}

results

{'a': [2, 4, 6],
 'b': [8, 6, 4],
 'c': [8, 6, 4],
 'd': [],
 'e': [3, 2, 1, 0],
 'f': [9, 6, 3, 0]}

### Explanation

`data = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]`

- `data[2:8:2]` visits indices `2, 4, 6`.
- `data[8:2:-2]` visits indices `8, 6, 4`.
- `data[-2:-8:-2]` becomes roughly `data[8:2:-2]`, so it visits indices `8, 6, 4`.
- `data[3:-1:-1]` is empty because `-1` normalizes to index `9`, producing the equivalent range `range(3, 9, -1)`, which cannot move from `3` toward `9` using a negative step.
- `data[3::-1]` visits indices `3, 2, 1, 0`.
- `data[::-3]` starts at the end and visits every third item backward: indices `9, 6, 3, 0`.

## Problem 2 — Convert Any Slice to the Indices It Visits

Write a function `visited_indices(s, length)` that accepts a `slice` object and a sequence length, and returns the exact list of indices that the slice would visit.

Requirements:

1. Use `slice.indices(length)`.
2. Support positive and negative steps.
3. Support omitted `start`, `stop`, and `step`.
4. Raise the same error Python raises when the step is zero.

In [2]:
# Solution 2

def visited_indices(s, length):
    """Return the concrete indices visited by slice s for a sequence of given length."""
    start, stop, step = s.indices(length)
    return list(range(start, stop, step))


# Checks
assert visited_indices(slice(None, None, None), 5) == [0, 1, 2, 3, 4]
assert visited_indices(slice(None, None, -1), 5) == [4, 3, 2, 1, 0]
assert visited_indices(slice(1, 10, 2), 6) == [1, 3, 5]
assert visited_indices(slice(3, -1, -1), 6) == []
assert visited_indices(slice(3, None, -1), 6) == [3, 2, 1, 0]

try:
    visited_indices(slice(None, None, 0), 5)
except ValueError as ex:
    zero_step_error = str(ex)

zero_step_error

'slice step cannot be zero'

### Explanation

The `.indices(length)` method normalizes a slice into a concrete `(start, stop, step)` tuple for a sequence of a given length.

Then `range(start, stop, step)` gives the exact indices visited by the slice.

## Problem 3 — Implement a Safe Window Extractor

Write a function `window(seq, center, radius)` that returns a centered window from `seq`.

For example:

```python
window([10, 20, 30, 40, 50], center=2, radius=1)
```

should return:

```python
[20, 30, 40]
```

Requirements:

1. Use slicing, not manual loops.
2. It should not raise `IndexError` if the window extends beyond either side.
3. It should support strings, tuples, and lists.
4. It should raise `ValueError` if `radius < 0`.

In [3]:
# Solution 3

def window(seq, center, radius):
    if radius < 0:
        raise ValueError("radius must be non-negative")
    return seq[center - radius:center + radius + 1]


# Checks
assert window([10, 20, 30, 40, 50], center=2, radius=1) == [20, 30, 40]
assert window([10, 20, 30], center=0, radius=5) == [10, 20, 30]
assert window("abcdef", center=2, radius=2) == "abcde"
assert window((1, 2, 3, 4), center=3, radius=2) == (2, 3, 4)

try:
    window([1, 2, 3], center=1, radius=-1)
except ValueError as ex:
    error_message = str(ex)

error_message

'radius must be non-negative'

### Explanation

Slicing is safer than direct indexing for boundary-heavy logic because out-of-bounds slice limits are clipped automatically.

`seq[center - radius:center + radius + 1]` includes the center element because the stop index is exclusive.

## Problem 4 — Fixed-Width Record Parser Using Named Slices

You receive fixed-width records where each row has the following layout:

| Field | Start | Stop |
|---|---:|---:|
| account_id | 0 | 6 |
| first_name | 6 | 18 |
| last_name | 18 | 30 |
| balance | 30 | 40 |
| status | 40 | 41 |

Write a parser that converts rows into dictionaries.

Requirements:

1. Define named `slice()` objects.
2. Strip whitespace from text fields.
3. Convert `balance` to `float`.
4. Convert status codes using this mapping:
   - `A` → `active`
   - `S` → `suspended`
   - `C` → `closed`

In [4]:
# Solution 4

ACCOUNT_ID = slice(0, 6)
FIRST_NAME = slice(6, 18)
LAST_NAME = slice(18, 30)
BALANCE = slice(30, 40)
STATUS = slice(40, 41)

STATUS_MAP = {
    "A": "active",
    "S": "suspended",
    "C": "closed",
}

def make_record(account_id, first_name, last_name, balance, status):
    return (
        f"{account_id:<6}"
        f"{first_name:<12}"
        f"{last_name:<12}"
        f"{balance:>10}"
        f"{status:<1}"
    )

def parse_record(row):
    status_code = row[STATUS].strip()

    return {
        "account_id": row[ACCOUNT_ID].strip(),
        "first_name": row[FIRST_NAME].strip(),
        "last_name": row[LAST_NAME].strip(),
        "balance": float(row[BALANCE].strip()),
        "status": STATUS_MAP[status_code],
    }


rows = [
    make_record("000123", "Ada", "Lovelace", "001250.50", "A"),
    make_record("000456", "Grace", "Hopper", "000300.00", "S"),
    make_record("000789", "Alan", "Turing", "009999.99", "C"),
]

parsed = [parse_record(row) for row in rows]
parsed

[{'account_id': '000123',
  'first_name': 'Ada',
  'last_name': 'Lovelace',
  'balance': 1250.5,
  'status': 'active'},
 {'account_id': '000456',
  'first_name': 'Grace',
  'last_name': 'Hopper',
  'balance': 300.0,
  'status': 'suspended'},
 {'account_id': '000789',
  'first_name': 'Alan',
  'last_name': 'Turing',
  'balance': 9999.99,
  'status': 'closed'}]

### Explanation

Named slices make fixed-width parsing easier to maintain.

If a column boundary changes later, only the slice definition needs to change, not every place where that field is extracted.

## Problem 5 — Build a Slice-Based Pagination Function

Write a function `paginate(seq, page, page_size)` that returns the items for a given page.

Rules:

1. Pages are 1-based.
2. `page_size` must be positive.
3. `page` must be positive.
4. Use slicing.
5. If the requested page is beyond the available data, return an empty sequence of the same type where possible.

In [5]:
# Solution 5

def paginate(seq, page, page_size):
    if page <= 0:
        raise ValueError("page must be positive")
    if page_size <= 0:
        raise ValueError("page_size must be positive")

    start = (page - 1) * page_size
    stop = start + page_size
    return seq[start:stop]


# Checks
items = list(range(1, 21))

assert paginate(items, page=1, page_size=5) == [1, 2, 3, 4, 5]
assert paginate(items, page=2, page_size=5) == [6, 7, 8, 9, 10]
assert paginate(items, page=4, page_size=6) == [19, 20]
assert paginate(items, page=99, page_size=5) == []
assert paginate("abcdefghij", page=2, page_size=3) == "def"

paginate(items, page=3, page_size=7)

[15, 16, 17, 18, 19, 20]

### Explanation

For page `p` and page size `n`, the starting index is `(p - 1) * n` and the stopping index is `start + n`.

Slicing automatically handles the final short page and pages beyond the end.

## Problem 6 — Chunk a Sequence Using Slicing

Write a function `chunks(seq, size)` that splits a sequence into consecutive chunks.

Example:

```python
chunks([1, 2, 3, 4, 5], 2)
```

should return:

```python
[[1, 2], [3, 4], [5]]
```

Requirements:

1. Use slicing.
2. Support lists, tuples, and strings.
3. Preserve the natural slice type of the input sequence.
4. Raise `ValueError` if `size <= 0`.

In [6]:
# Solution 6

def chunks(seq, size):
    if size <= 0:
        raise ValueError("size must be positive")
    return [seq[i:i + size] for i in range(0, len(seq), size)]


# Checks
assert chunks([1, 2, 3, 4, 5], 2) == [[1, 2], [3, 4], [5]]
assert chunks((1, 2, 3, 4, 5), 2) == [(1, 2), (3, 4), (5,)]
assert chunks("abcdefg", 3) == ["abc", "def", "g"]

try:
    chunks([1, 2, 3], 0)
except ValueError as ex:
    chunk_error = str(ex)

chunk_error

'size must be positive'

### Explanation

`range(0, len(seq), size)` gives the starting index of each chunk.

Each chunk is produced by `seq[i:i + size]`. The final chunk may be shorter, and slicing handles that automatically.

## Problem 7 — Implement a Custom Sequence with Slice Support

Create a class `LoggedList` that wraps a list and supports indexing and slicing.

Requirements:

1. `obj[i]` should return a single item.
2. `obj[start:stop:step]` should return a new `LoggedList`.
3. Every access should be recorded in `obj.history`.
4. Slice access should store the normalized indices visited by the slice.
5. Use `slice.indices()` for normalization.

In [7]:
# Solution 7

class LoggedList:
    def __init__(self, data):
        self._data = list(data)
        self.history = []

    def __getitem__(self, key):
        if isinstance(key, slice):
            normalized = key.indices(len(self._data))
            visited = list(range(*normalized))
            self.history.append({
                "type": "slice",
                "original": key,
                "normalized": normalized,
                "visited": visited,
            })
            return LoggedList(self._data[key])

        self.history.append({
            "type": "index",
            "index": key,
        })
        return self._data[key]

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"LoggedList({self._data!r})"

    def to_list(self):
        return self._data.copy()


# Checks
x = LoggedList([10, 20, 30, 40, 50, 60])

assert x[2] == 30
y = x[::-2]

assert isinstance(y, LoggedList)
assert y.to_list() == [60, 40, 20]
assert x.history[0] == {"type": "index", "index": 2}
assert x.history[1]["type"] == "slice"
assert x.history[1]["normalized"] == (5, -1, -2)
assert x.history[1]["visited"] == [5, 3, 1]

x.history

[{'type': 'index', 'index': 2},
 {'type': 'slice',
  'original': slice(None, None, -2),
  'normalized': (5, -1, -2),
  'visited': [5, 3, 1]}]

### Explanation

Inside `__getitem__`, Python passes either an integer index or a `slice` object.

Using `isinstance(key, slice)` lets the class distinguish between normal indexing and slicing.

Returning a new `LoggedList` for slices mimics the way built-in list slicing returns a new list.

## Problem 8 — Validate Slice Equivalence

Write a function `slice_equals_manual(seq, s)` that verifies whether slicing `seq[s]` gives the same result as manually collecting items from the indices produced by `s.indices(len(seq))`.

Requirements:

1. Use `slice.indices()`.
2. Use `range()` to generate visited indices.
3. Return `True` if both approaches match.
4. Test it against tricky positive and negative slices.

In [8]:
# Solution 8

def slice_equals_manual(seq, s):
    start, stop, step = s.indices(len(seq))
    manual = [seq[i] for i in range(start, stop, step)]
    sliced = seq[s]

    # Convert sliced to list so this works uniformly for strings, tuples, and lists.
    return list(sliced) == manual


seq = list("abcdefghij")

test_slices = [
    slice(None, None, None),
    slice(None, None, -1),
    slice(2, 8, 2),
    slice(8, 2, -2),
    slice(-100, 100, 3),
    slice(3, -1, -1),
    slice(3, None, -1),
    slice(None, None, -3),
]

results = [(s, slice_equals_manual(seq, s)) for s in test_slices]

assert all(result for _, result in results)
results

[(slice(None, None, None), True),
 (slice(None, None, -1), True),
 (slice(2, 8, 2), True),
 (slice(8, 2, -2), True),
 (slice(-100, 100, 3), True),
 (slice(3, -1, -1), True),
 (slice(3, None, -1), True),
 (slice(None, None, -3), True)]

### Explanation

A slice can be understood as a normalized range of indices.

For any sequence `seq` and slice `s`, this idea is essentially:

```python
seq[s] == [seq[i] for i in range(*s.indices(len(seq)))]
```

The exact container type may differ, so the solution compares list forms.

## Problem 9 — Redact Sensitive Text Using Slice Assignment

Given a list of characters, redact a sensitive region using slice assignment.

Write a function `redact(chars, start, stop, replacement='*')`.

Requirements:

1. `chars` is a list of single-character strings.
2. Replace the selected region with the replacement character repeated to match the region length.
3. Modify the list in place.
4. Return the same list object.
5. Use slicing or slice assignment.

In [9]:
# Solution 9

def redact(chars, start, stop, replacement="*"):
    selected_length = len(chars[start:stop])
    chars[start:stop] = [replacement] * selected_length
    return chars


# Checks
text = list("SSN: 123-45-6789")
original_id = id(text)

result = redact(text, 5, 16)

assert id(result) == original_id
assert "".join(result) == "SSN: ***********"

"".join(text)

'SSN: ***********'

### Explanation

List slice assignment can replace a region in place.

This is different from ordinary slicing, which creates a new list.

The expression `chars[start:stop]` safely determines how many characters are actually selected, even when `start` or `stop` is out of bounds.

## Problem 10 — Rotate a Sequence Using Slicing

Write a function `rotate(seq, k)` that rotates a sequence by `k` positions.

Rules:

1. Positive `k` rotates to the right.
2. Negative `k` rotates to the left.
3. Use slicing.
4. Support lists, tuples, and strings.
5. Empty sequences should return themselves unchanged.

In [10]:
# Solution 10

def rotate(seq, k):
    n = len(seq)
    if n == 0:
        return seq

    k = k % n
    return seq[-k:] + seq[:-k]


# Checks
assert rotate([1, 2, 3, 4, 5], 2) == [4, 5, 1, 2, 3]
assert rotate([1, 2, 3, 4, 5], -1) == [2, 3, 4, 5, 1]
assert rotate("abcdef", 2) == "efabcd"
assert rotate((1, 2, 3), 4) == (3, 1, 2)
assert rotate([], 10) == []

rotate("slicing", 3)

'ingslic'

### Explanation

The modulo operation normalizes the rotation amount so large rotations still work.

For a right rotation by `k`, the last `k` items move to the front:

```python
seq[-k:] + seq[:-k]
```

Because slicing preserves the sequence type for strings, tuples, and lists, the function works naturally across all three.

## Problem 11 — Every Nth Item from the End

Write a function `from_end_every(seq, n)` that returns every `n`th item while traversing from the end toward the beginning.

Requirements:

1. Use slicing.
2. `n` must be positive.
3. The function should work for lists, tuples, and strings.
4. Do not call `reversed()`.

In [11]:
# Solution 11

def from_end_every(seq, n):
    if n <= 0:
        raise ValueError("n must be positive")
    return seq[::-n]


# Checks
assert from_end_every([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 3) == [9, 6, 3, 0]
assert from_end_every("abcdefg", 2) == "geca"
assert from_end_every((1, 2, 3, 4, 5), 2) == (5, 3, 1)

try:
    from_end_every([1, 2, 3], 0)
except ValueError as ex:
    n_error = str(ex)

n_error

'n must be positive'

### Explanation

`seq[::-n]` means:

- start from the default reverse start position,
- continue to the default reverse stop position,
- move backward by `n` each time.

For example, `seq[::-3]` visits the last item, then three positions back, and so on.

## Problem 12 — Diagnose an Empty Negative-Step Slice

Write a function `diagnose_slice(seq, s)` that returns a dictionary containing:

1. The original slice.
2. The normalized `(start, stop, step)` tuple.
3. The visited indices.
4. The sliced result.
5. Whether the result is empty.

Use it to explain why `seq[3:-1:-1]` is empty while `seq[3::-1]` is not.

In [12]:
# Solution 12

def diagnose_slice(seq, s):
    normalized = s.indices(len(seq))
    visited = list(range(*normalized))
    result = seq[s]

    return {
        "original": s,
        "normalized": normalized,
        "visited_indices": visited,
        "result": result,
        "is_empty": len(result) == 0,
    }


seq = [0, 1, 2, 3, 4, 5]

case_1 = diagnose_slice(seq, slice(3, -1, -1))
case_2 = diagnose_slice(seq, slice(3, None, -1))

case_1, case_2

({'original': slice(3, -1, -1),
  'normalized': (3, 5, -1),
  'visited_indices': [],
  'result': [],
  'is_empty': True},
 {'original': slice(3, None, -1),
  'normalized': (3, -1, -1),
  'visited_indices': [3, 2, 1, 0],
  'result': [3, 2, 1, 0],
  'is_empty': False})

### Explanation

`slice(3, -1, -1).indices(6)` normalizes to `(3, 5, -1)`, so the visited range is:

```python
range(3, 5, -1)
```

That range is empty because it tries to move from `3` toward `5` using a negative step.

`slice(3, None, -1).indices(6)` normalizes to `(3, -1, -1)`, so the visited range is:

```python
range(3, -1, -1)
```

That visits indices `3, 2, 1, 0`.

# Summary

Key advanced slicing ideas practiced in this notebook:

- `slice()` creates reusable slice objects.
- Slice start is inclusive; stop is exclusive.
- Slice bounds can safely go out of range.
- Negative steps traverse backward.
- Some negative-step slices are empty because the normalized range cannot progress.
- `slice.indices(length)` reveals the effective `(start, stop, step)`.
- Slicing is useful for parsing, pagination, chunking, redaction, rotation, and custom sequence types.